In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abdullahnaeem151015/automotive-price-prediction-dataset/vehicle_price_prediction (1).csv


In [2]:
#Loading the dataset
import pandas as pd
import numpy as np

file_path = "/kaggle/input/datasets/abdullahnaeem151015/automotive-price-prediction-dataset/vehicle_price_prediction (1).csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000000, 20)


,make,model,year,mileage,engine_hp,transmission,fuel_type,drivetrain,body_type,exterior_color,interior_color,owner_count,accident_history,seller_type,condition,trim,vehicle_age,mileage_per_year,brand_popularity,price
0,Volkswagen,Jetta,2016,183903,173,Manual,Electric,RWD,Sedan,Blue,Brown,5,NaN,Dealer,Excellent,EX,9,20433.666667,0.040054,7208.52
1,Lexus,RX,2010,236643,352,Manual,Gasoline,FWD,Sedan,Silver,Beige,5,Minor,Dealer,Good,LX,15,15776.200000,0.039921,6911.81
2,Subaru,Crosstrek,2016,103199,188,Automatic,Diesel,AWD,Sedan,Silver,Beige,5,NaN,Dealer,Excellent,Touring,9,11466.555556,0.040230,11915.63
3,Cadillac,Lyriq,2016,118889,338,Manual,Gasoline,AWD,SUV,Black,Gray,3,NaN,Private,Good,Base,9,13209.888889,0.039847,25984.79
4,Toyota,Highlander,2018,204170,196,Manual,Diesel,FWD,Sedan,Red,Brown,5,Minor,Dealer,Excellent,Sport,7,29167.142857,0.039627,8151.30


In [3]:
# Understand the Prediction Target
df['price'].describe()

count    1000000.000000
mean       20329.302593
std        13644.467189
min         1500.000000
25%        10324.527500
50%        17864.740000
75%        27601.402500
max        93422.090000
Name: price, dtype: float64

In [3]:
#Inspecting all the column's datatype
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Columns:
['make', 'model', 'year', 'mileage', 'engine_hp', 'transmission', 'fuel_type', 'drivetrain', 'body_type', 'exterior_color', 'interior_color', 'owner_count', 'accident_history', 'seller_type', 'condition', 'trim', 'vehicle_age', 'mileage_per_year', 'brand_popularity', 'price']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 20 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   make              1000000 non-null  object 
 1   model             1000000 non-null  object 
 2   year              1000000 non-null  int64  
 3   mileage           1000000 non-null  int64  
 4   engine_hp         1000000 non-null  int64  
 5   transmission      1000000 non-null  object 
 6   fuel_type         1000000 non-null  object 
 7   drivetrain        1000000 non-null  object 
 8   body_type         1000000 non-null  object 
 9   exterior_color    1000000 non-null  

In [4]:
#Inspecting basic statistics of numerical data columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
year,1000000.0,2017.453816,3.902285,2000.000000,2015.000000,2018.000000,2020.000000,2025.000000
mileage,1000000.0,112660.126464,72088.236925,500.000000,57654.000000,103331.000000,157865.000000,300000.000000
engine_hp,1000000.0,235.664301,93.614773,90.000000,162.000000,215.000000,300.000000,581.000000
owner_count,1000000.0,3.477846,1.544856,1.000000,2.000000,4.000000,5.000000,5.000000
vehicle_age,1000000.0,7.585987,3.829534,1.000000,5.000000,7.000000,10.000000,25.000000
mileage_per_year,1000000.0,14540.674200,6108.901522,33.333333,10487.633523,14688.118056,18685.942308,55391.000000
brand_popularity,1000000.0,0.040001,0.000234,0.039318,0.039842,0.040022,0.040148,0.040484
price,1000000.0,20329.302593,13644.467189,1500.000000,10324.527500,17864.740000,27601.402500,93422.090000


In [5]:
#Inspecting basic statistics of categorical datatype columns
df.describe(include="object").T

,count,unique,top,freq
make,1000000,25,Kia,40484
model,1000000,105,300,20202
transmission,1000000,2,Automatic,500263
fuel_type,1000000,3,Electric,359597
drivetrain,1000000,3,AWD,361110
body_type,1000000,7,SUV,406693
exterior_color,1000000,6,White,166934
interior_color,1000000,4,Beige,250570
accident_history,249867,2,Minor,199981
seller_type,1000000,2,Dealer,699710


In [6]:
#Checking whether missing values exist in the dataset
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

missing.sort_values("Missing %", ascending=False)

,Missing Count,Missing %
accident_history,750133,75.01
make,0,0.00
year,0,0.00
model,0,0.00
mileage,0,0.00
engine_hp,0,0.00
fuel_type,0,0.00
transmission,0,0.00
body_type,0,0.00
exterior_color,0,0.00


In [7]:
#Checking whether vehicle's milage, age, year it was manufactured, engine hp has relation with price
df.select_dtypes(include=np.number).corr()["price"].sort_values(ascending=False)


price               1.000000
year                0.664730
engine_hp           0.653339
brand_popularity    0.055940
mileage_per_year   -0.216451
owner_count        -0.449437
mileage            -0.618349
vehicle_age        -0.662811
Name: price, dtype: float64

In [8]:
import numpy as np
import pandas as pd


# ==========================================
# CORRELATION RATIO (ETA)
# ==========================================

def correlation_ratio(categories, values):
    
    # Remove rows where category or price is missing
    valid = categories.notna() & values.notna()
    
    categories = categories[valid]
    values = values[valid]

    # Convert categories into numeric group IDs
    category_codes, _ = pd.factorize(categories)

    # Overall mean price
    overall_mean = values.mean()

    # Variation explained by categories
    between_group_variation = 0

    # Total variation in price
    total_variation = np.sum(
        (values - overall_mean) ** 2
    )

    # Calculate variation for each category
    for category_id in np.unique(category_codes):

        group_values = values[
            category_codes == category_id
        ]

        if len(group_values) == 0:
            continue

        group_mean = group_values.mean()

        between_group_variation += (
            len(group_values)
            *
            (group_mean - overall_mean) ** 2
        )

    # Avoid division by zero
    if total_variation == 0:
        return 0.0

    eta_squared = (
        between_group_variation
        /
        total_variation
    )

    # Return eta rather than eta-squared
    return np.sqrt(eta_squared)


# ==========================================
# FIND CATEGORICAL COLUMNS
# ==========================================

categorical_columns = (
    df.select_dtypes(
        include=['object', 'category']
    )
    .columns
)


# ==========================================
# CALCULATE RELATIONSHIP WITH PRICE
# ==========================================

categorical_correlations = {}

for column in categorical_columns:

    categorical_correlations[column] = (
        correlation_ratio(
            df[column],
            df['price']
        )
    )


# ==========================================
# SORT STRONGEST → WEAKEST
# ==========================================

categorical_correlations = (
    pd.Series(
        categorical_correlations,
        name='correlation_with_price'
    )
    .sort_values(
        ascending=False
    )
)


categorical_correlations

model               0.674928
make                0.674902
body_type           0.204533
accident_history    0.101918
condition           0.064413
fuel_type           0.049363
seller_type         0.033468
drivetrain          0.029648
interior_color      0.003506
trim                0.002540
exterior_color      0.001686
transmission        0.001389
Name: correlation_with_price, dtype: float64

In [10]:
#Checking correlation of vehicle condition with price
df.groupby("condition")["price"].median().sort_values(ascending=False)

condition
Excellent    18593.63
Good         17718.60
Fair         15963.25
Name: price, dtype: float64

In [ ]:
#Checking duplicates
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:",
      round(duplicate_count / len(df) * 100, 2), "%")

In [ ]:
#Examining categorical values and checking their uniques values
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
#Inspecting numerical data column ranges to understand are the values written in them are real
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    print(
        col,
        "| Min:", df[col].min(),
        "| Max:", df[col].max(),
        "| Median:", df[col].median()
    )

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 20 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   make              1000000 non-null  object 
 1   model             1000000 non-null  object 
 2   year              1000000 non-null  int64  
 3   mileage           1000000 non-null  int64  
 4   engine_hp         1000000 non-null  int64  
 5   transmission      1000000 non-null  object 
 6   fuel_type         1000000 non-null  object 
 7   drivetrain        1000000 non-null  object 
 8   body_type         1000000 non-null  object 
 9   exterior_color    1000000 non-null  object 
 10  interior_color    1000000 non-null  object 
 11  owner_count       1000000 non-null  int64  
 12  accident_history  249867 non-null   object 
 13  seller_type       1000000 non-null  object 
 14  condition         1000000 non-null  object 
 15  trim              1000000 non-null  object 
 16  v

In [15]:
#Replacing the missing values from accident history column with unknown 
df_clean = df.copy()

df_clean["accident_history"] = (
    df_clean["accident_history"].fillna("Unknown")
)

df_clean["accident_history"].value_counts()

accident_history
Unknown    750133
Minor      199981
Major       49886
Name: count, dtype: int64

In [16]:
# Checking correlations between predictor themselves
df[
    ["year", "vehicle_age", "mileage",
     "mileage_per_year", "engine_hp",
     "owner_count", "brand_popularity"]
].corr().round(3)

,year,vehicle_age,mileage,mileage_per_year,engine_hp,owner_count,brand_popularity
year,1.000,-0.999,-0.784,-0.116,0.000,-0.649,-0.000
vehicle_age,-0.999,1.000,0.783,0.101,-0.000,0.646,0.000
mileage,-0.784,0.783,1.000,0.603,0.000,0.514,-0.001
mileage_per_year,-0.116,0.101,0.603,1.000,-0.000,0.094,-0.003
engine_hp,0.000,-0.000,0.000,-0.000,1.000,0.000,0.077
owner_count,-0.649,0.646,0.514,0.094,0.000,1.000,-0.001
brand_popularity,-0.000,0.000,-0.001,-0.003,0.077,-0.001,1.000


In [17]:
# Categorical features to compare with vehicle price
categorical_features = [
    "make",
    "model",
    "transmission",
    "fuel_type",
    "drivetrain",
    "body_type",
    "exterior_color",
    "interior_color",
    "accident_history",
    "seller_type",
    "condition",
    "trim"
]
# Compare all categorical features with vehicle price

for col in categorical_features:
    print(f"\n{'='*50}")
    print(f"{col.upper()} vs PRICE")
    print(f"{'='*50}")

    result = (
        df.groupby(col, dropna=False)["price"]
          .agg(
              count="count",
              mean_price="mean",
              median_price="median"
          )
          .sort_values("median_price", ascending=False)
    )

    print(result.head(20))


MAKE vs PRICE
               count    mean_price  median_price
make                                            
Porsche        40210  43004.618981     41352.590
Land Rover     40133  39501.134438     37832.970
Tesla          40226  32419.062274     31175.835
Cadillac       39847  30385.456083     29237.300
Mercedes-Benz  39794  29122.907672     27911.420
Audi           40022  27611.112472     26531.815
Volvo          39842  26963.491616     25976.955
BMW            39840  25483.833390     24497.530
Lexus          39921  24092.952386     23112.510
Acura          40147  21924.884587     21067.030
GMC            39754  19205.377165     18484.695
Ram            40148  18553.443497     17882.405
Chrysler       40059  17827.852912     17071.840
Jeep           40006  16549.233887     15899.140
Dodge          40035  15767.936428     15131.000
Chevrolet      39982  15116.149275     14482.075
Ford           39842  13809.840171     13173.195
Subaru         40230  13130.184969     12561.680
Volks

In [18]:
#Installing important machine learning libraries for training my data and giving future predictions
!pip install -q xgboost lightgbm catboost

In [19]:
#Importing all the important libraries that my models will use while training and giving predictions
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.ensemble import VotingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print("All imports loaded successfully")

All imports loaded successfully


In [20]:
# Keep original dataframe safe
model_df = df_clean.copy()

# Missing accident history means unknown, NOT "no accident"
model_df["accident_history"] = (
    model_df["accident_history"].fillna("Unknown")
)
#All features for my model to train on to learn historical patterns
features = [
    "make",
    "model",
    "mileage",
    "engine_hp",
    "transmission",
    "fuel_type",
    "drivetrain",
    "body_type",
    "exterior_color",
    "interior_color",
    "owner_count",
    "accident_history",
    "seller_type",
    "condition",
    "trim",
    "vehicle_age",
    "mileage_per_year"
]

#Using train/test split to select 80 % of features for model training and 20 % for testing
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.20,
    random_state=42
)

# Create X = input features
X = model_df[features].copy()

# Create y = target we want to predict
y = model_df["price"].copy()

# Create actual training/testing datasets
X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]


# Verify everything was created correctly
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTraining:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting:")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X shape: (1000000, 17)
y shape: (1000000,)

Training:
X_train: (800000, 17)
y_train: (800000,)

Testing:
X_test : (200000, 17)
y_test : (200000,)


In [21]:
# ============================================================
# TARGET ENCODING PREPROCESSOR
# ============================================================

def create_target_preprocessor(X):

    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numerical_cols = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),

        (
            "encoder",
            TargetEncoder(
                target_type="continuous",
                smooth="auto",
                cv=5,
                random_state=42
            )
        )
    ])

    # Numerical preprocessing
    numerical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "cat",
            categorical_pipeline,
            categorical_cols
        ),
        (
            "num",
            numerical_pipeline,
            numerical_cols
        )
    ])

    return preprocessor

In [22]:
#  Define the Candidate Models

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

def create_models():
    # 1. XGBoost: Deep and highly regularized
    xgb = XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,          # Well-suited for complex target-encoded combinations
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    # 2. LightGBM: Increased capacity to match the depth of the others
    lgbm = LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,          # Restricts maximum tree depth explicitly
        num_leaves=64,         # Increased from 31 to capture more variance at 1M rows
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    # 3. CatBoost: Strong out-of-the-box performance
    catboost = CatBoostRegressor(
        iterations=400,
        learning_rate=0.05,
        depth=8,              # Matches the structural complexity of XGB/LGBM
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        thread_count=-1
    )

    return xgb, lgbm, catboost

In [23]:
#Evaluating model performance using metrices like mae, rmse, r2
def evaluate_model(name, model, X_train, X_test, y_train, y_test):

    start = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_sec": training_time
    }

In [24]:
# ============================================================
# FULL EXPERIMENT 
# TARGET ENCODING + BEST FEATURE SET
# ============================================================

def run_target_encoding_experiment():

    results = []

    xgb, lgbm, catboost = create_models()

    models = {
        "XGBoost": xgb,
        "LightGBM": lgbm,
        "CatBoost": catboost
    }

    # --------------------------------
    # Individual models
    # --------------------------------

    for name, regressor in models.items():

        pipeline = Pipeline([
            (
                "preprocessor",
                create_target_preprocessor(X_train)
            ),
            (
                "model",
                regressor
            )
        ])

        result = evaluate_model(
            name,
            pipeline,
            X_train,
            X_test,
            y_train,
            y_test
        )

        result["Experiment"] = "Experiment - Target Encoding"

        results.append(result)

    # --------------------------------
    # Voting Regressor
    # --------------------------------

    xgb_vote, lgbm_vote, cat_vote = create_models()

    voting = VotingRegressor(
        estimators=[
            ("xgb", xgb_vote),
            ("lgbm", lgbm_vote),
            ("cat", cat_vote)
        ]
    )

    voting_pipeline = Pipeline([
        (
            "preprocessor",
            create_target_preprocessor(X_train)
        ),
        (
            "model",
            voting
        )
    ])

    voting_result = evaluate_model(
        "Voting Regressor",
        voting_pipeline,
        X_train,
        X_test,
        y_train,
        y_test
    )

    voting_result["Experiment"] = (
        "Experiment  - Target Encoding"
    )

    results.append(voting_result)

    return pd.DataFrame(results), voting_pipeline 

In [25]:
##  Compare Model Performance
experiment_results, voting_pipeline = (
    run_target_encoding_experiment()
)

experiment_results.round(3)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Model,MAE,RMSE,R2,Training_Time_sec,Experiment
0,XGBoost,833.706,1240.519,0.992,26.643,Experiment - Target Encoding
1,LightGBM,843.903,1245.578,0.992,24.364,Experiment - Target Encoding
2,CatBoost,835.689,1235.633,0.992,44.219,Experiment - Target Encoding
3,Voting Regressor,827.404,1231.647,0.992,84.629,Experiment - Target Encoding


In [26]:
# ============================================================
# PERMUTATION IMPORTANCE FOR FINAL VOTING PIPELINE
# ============================================================

import pandas as pd

from sklearn.inspection import permutation_importance


# ------------------------------------------------------------
# 1. Use a manageable sample of the test set
# ------------------------------------------------------------
# Your dataset is very large, so running permutation importance
# on the full X_test can take a long time.
#
# 100,000 rows is usually enough for a useful importance ranking.

sample_size = min(
    100000,
    len(X_test)
)

X_importance = X_test.sample(
    n=sample_size,
    random_state=42
)

y_importance = y_test.loc[
    X_importance.index
]


print(
    "Permutation importance rows:",
    len(X_importance)
)


# ------------------------------------------------------------
# 2. Calculate permutation importance
# ------------------------------------------------------------
# scoring="neg_mean_absolute_error"
#
# This asks:
# "How much worse does MAE become when this feature is shuffled?"
#
# A larger importance means the model relies more strongly
# on that feature.

permutation_result = permutation_importance(

    voting_pipeline,

    X_importance,
    y_importance,

    scoring="neg_mean_absolute_error",

    n_repeats=5,

    random_state=42,

    n_jobs=-1
)


# ------------------------------------------------------------
# 3. Create clean results table
# ------------------------------------------------------------

importance_df = pd.DataFrame({

    "Feature":
        X_importance.columns,

    "Importance":
        permutation_result.importances_mean,

    "Std_Dev":
        permutation_result.importances_std

})

 
# ------------------------------------------------------------
# 4. Sort most important → least important
# ------------------------------------------------------------

importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Add relative importance percentage
# ------------------------------------------------------------

positive_importance = (
    importance_df["Importance"]
    .clip(lower=0)
)

total_importance = (
    positive_importance.sum()
)


if total_importance > 0:

    importance_df[
        "Relative_Importance_%"
    ] = (
        positive_importance
        /
        total_importance
        *
        100
    )

else:

    importance_df[
        "Relative_Importance_%"
    ] = 0


# ------------------------------------------------------------
# 6. Display clean table
# ------------------------------------------------------------

importance_df.round(4)

Permutation importance rows: 100000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

,Feature,Importance,Std_Dev,Relative_Importance_%
0,vehicle_age,6693.9124,13.6762,34.7681
1,make,5158.5959,12.1215,26.7937
2,mileage,2992.8318,5.6814,15.5447
3,model,1635.8514,5.9049,8.4966
4,engine_hp,1388.2604,5.7392,7.2106
5,accident_history,767.2081,2.6808,3.9849
6,condition,437.4082,1.5723,2.2719
7,seller_type,144.6918,0.8821,0.7515
8,mileage_per_year,32.7104,0.1646,0.1699
9,owner_count,1.2824,0.0590,0.0067
